In [1]:
## Download lib
%pip install tokenizers pypdf

Note: you may need to restart the kernel to use updated packages.


In [2]:
## PDF'leri text formatına çevirip tek bir corpus'ta birleştir
from pypdf import PdfReader

PDF_PATHS = [
    "nutuk.pdf",
    "09162252_jules-verne-aya-yolculuk.pdf",
    "15182733_Jules-Verne-Denizler-Altinda-20-Bin-Fersah.pdf",
]
TXT_PATH = "corpus.txt"

tum_metinler = []
for pdf_yolu in PDF_PATHS:
    reader = PdfReader(pdf_yolu)
    sayfa_metinleri = [sayfa.extract_text() or "" for sayfa in reader.pages]
    metin = "\n".join(sayfa_metinleri)
    tum_metinler.append(metin)
    print(f"{pdf_yolu}: {len(reader.pages)} sayfa, {len(metin)} karakter")

corpus_metni = "\n\n".join(tum_metinler)

with open(TXT_PATH, "w", encoding="utf-8") as f:
    f.write(corpus_metni)

print(f"\nToplam karakter: {len(corpus_metni)}")
print("--- İlk 200 karakter ---")
print(corpus_metni[:200])

nutuk.pdf: 430 sayfa, 1521466 karakter
09162252_jules-verne-aya-yolculuk.pdf: 158 sayfa, 319778 karakter
15182733_Jules-Verne-Denizler-Altinda-20-Bin-Fersah.pdf: 124 sayfa, 175197 karakter

Toplam karakter: 2016445
--- İlk 200 karakter ---
1
NUTUK
Mustafa Kemal ATATÜRK
SAMSUN'A ÇIKTIĞIM GÜN GENEL DURUM VE GÖRÜNÜŞ
1919 yılı Mayısının 19'uncu günü Samsun'a çıktım. Ülkenin genel durumu ve görünüşü şöyledir :
Osmanlı Devleti'nin içinde bulu


In [3]:
## Input folder
DATA_PATH = "corpus.txt"

# Dosyanın var olup olmadığını ve kaç karakter içerdiğini kontrol edelim
with open(DATA_PATH, "r", encoding="utf-8") as f:
    text_preview = f.read()

print(f"File length: {len(text_preview)} characters")
print("--- First 200 characters ---")
print(text_preview[:200])

File length: 2016445 characters
--- First 200 characters ---
1
NUTUK
Mustafa Kemal ATATÜRK
SAMSUN'A ÇIKTIĞIM GÜN GENEL DURUM VE GÖRÜNÜŞ
1919 yılı Mayısının 19'uncu günü Samsun'a çıktım. Ülkenin genel durumu ve görünüşü şöyledir :
Osmanlı Devleti'nin içinde bulu


In [4]:
## Target tokenizer vocab size
VOCAB_SIZE = 256000
## Mın # of token occurrences to be included in the vocab (2 is default value bcs of BYTE :) )
MIN_FREQUENCY = 2          # bir token'ın vocab'a girmesi için minimum görülme sıklığı

In [5]:
## Import lib and create tokenizer object
from tokenizers import ByteLevelBPETokenizer
tokenizer = ByteLevelBPETokenizer()

In [6]:
## Özel durumlar için gerekli karakter tanımlamaları bir token olarak sisteme tanımlanır.
special_tokens = ["<s>", "<pad>", "</s>", "<unk>", "<mask>"]

## Oluşturulan tokenizer nesnesi ile verilen parametreler ile eğitim.
tokenizer.train(
    files=[DATA_PATH],
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=special_tokens,
)

print("Train Done")
print(f"Real vocab size: {tokenizer.get_vocab_size()}")




Train Done
Real vocab size: 33834


In [7]:
import os
import json as _json

OUTPUT_DIR = "bpe_tokenizer_256k"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Standart save_model (vocab.json + merges.txt) - tek satır sıkışık json üretir
tokenizer.save_model(OUTPUT_DIR)

# vocab.json'u id'ye göre sıralı, her token kendi satırında olacak şekilde yeniden yaz
vocab = tokenizer.get_vocab()
sorted_vocab = dict(sorted(vocab.items(), key=lambda x: x[1]))

with open(f"{OUTPUT_DIR}/vocab.json", "w", encoding="utf-8") as f:
    _json.dump(sorted_vocab, f, ensure_ascii=False, indent=1)

print(f"Saved: {OUTPUT_DIR}/vocab.json (satır satır) ve {OUTPUT_DIR}/merges.txt")

Saved: bpe_tokenizer_256k/vocab.json (satır satır) ve bpe_tokenizer_256k/merges.txt


In [8]:
## Hugging Face'e yükle
%pip install huggingface_hub

from huggingface_hub import notebook_login, HfApi, create_repo

notebook_login()  # HF token'ını burada gireceksin (write yetkili)

REPO_ID = "SalihHub/turkce-edebiyat-bpe-tokenizer-256k"

create_repo(REPO_ID, repo_type="model", exist_ok=True)

api = HfApi()
api.upload_folder(
    repo_id=REPO_ID,
    folder_path=OUTPUT_DIR,
    repo_type="model",
    commit_message=f"BPE tokenizer (vocab_size={VOCAB_SIZE}, {len(PDF_PATHS)} kitaplık corpus ile eğitildi)",
)

print(f"Yüklendi: https://huggingface.co/{REPO_ID}")

Note: you may need to restart the kernel to use updated packages.


/Users/salih/Desktop/magibu-work/magibu/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Yüklendi: https://huggingface.co/SalihHub/turkce-edebiyat-bpe-tokenizer-256k
